In [5]:
import torch
import cv2
import easyocr
import numpy as np
from ultralytics import YOLO
import requests

# 1. Download model file from Hugging Face
url = "https://huggingface.co/MKgoud/License-Plate-Recognizer/resolve/main/LP-detection.pt"
model_path = "LP-detection.pt"

r = requests.get(url)
with open(model_path, "wb") as f:
    f.write(r.content)

# 2. Load YOLO model
model = YOLO(model_path)

# 3. Load image
image_path = "lic.jpg"  # change to your image
img = cv2.imread(image_path)

# 4. Detect license plates
results = model(image_path)

# 5. OCR setup
reader = easyocr.Reader(['en'])

for r in results:
    boxes = r.boxes.xyxy.cpu().numpy().astype(int)
    for (x_min, y_min, x_max, y_max) in boxes:
        # Crop plate
        plate_crop = img[y_min:y_max, x_min:x_max]

        # OCR
        ocr_result = reader.readtext(plate_crop)
        if ocr_result:
            print("Detected Plate Text:", ocr_result[0][1])

        # Save cropped plate
        cv2.imwrite("cropped_plate.jpg", plate_crop)



image 1/1 C:\Users\H.T\Desktop\AI Lab\5.0 image vision\yolo\lic.jpg: 384x640 4 single_number_plates, 162.0ms
Speed: 5.4ms preprocess, 162.0ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Detected Plate Text: AFR-2u22]
Detected Plate Text: AFR-202
Detected Plate Text: [AFR2022
Detected Plate Text: AFR 021


In [4]:
import os
os.environ["YOLO_DISABLE_PATCHES"] = "1"

import cv2
import easyocr
from ultralytics import YOLO
import requests

# restore OpenCV GUI functions in case ultralytics patched them
cv2.imshow = cv2.__dict__['imshow']
cv2.destroyAllWindows = cv2.__dict__['destroyAllWindows']

# === 1. Download model from Hugging Face ===
url = "https://huggingface.co/MKgoud/License-Plate-Recognizer/resolve/main/LP-detection.pt"
model_path = "LP-detection.pt"

r = requests.get(url)
with open(model_path, "wb") as f:
    f.write(r.content)

# === 2. Load YOLO model ===
model = YOLO(model_path)

# === 3. Initialize OCR ===
reader = easyocr.Reader(['en'])

# === 4. Start webcam capture ===
cap = cv2.VideoCapture(0)  # change to 1 or 2 if you have multiple webcams

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO detection
    results = model(frame)

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy().astype(int)
        for (x_min, y_min, x_max, y_max) in boxes:
            # Draw bounding box
            cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

            # Crop plate
            plate_crop = frame[y_min:y_max, x_min:x_max]

            # OCR on plate
            ocr_result = reader.readtext(plate_crop)
            if ocr_result:
                plate_text = ocr_result[0][1]
                cv2.putText(frame, plate_text, (x_min, y_min - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
                print("Detected Plate Text:", plate_text)

    cv2.imshow("License Plate Detection", frame)

    # Exit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.



0: 480x640 (no detections), 205.6ms
Speed: 119.5ms preprocess, 205.6ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 187.0ms
Speed: 5.4ms preprocess, 187.0ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 168.5ms
Speed: 2.8ms preprocess, 168.5ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 198.9ms
Speed: 2.5ms preprocess, 198.9ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 177.6ms
Speed: 4.2ms preprocess, 177.6ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 201.7ms
Speed: 4.7ms preprocess, 201.7ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 205.1ms
Speed: 2.7ms preprocess, 205.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 149.2ms
Speed: 2.3ms pre

In [5]:
import cv2
import easyocr
from ultralytics import YOLO
import requests
import os

# === 1. Download model from Hugging Face if not already present ===
url = "https://huggingface.co/MKgoud/License-Plate-Recognizer/resolve/main/LP-detection.pt"
model_path = "LP-detection.pt"

if not os.path.exists(model_path):
    r = requests.get(url)
    with open(model_path, "wb") as f:
        f.write(r.content)

# === 2. Load YOLO model ===
model = YOLO(model_path)

# === 3. Initialize OCR ===
reader = easyocr.Reader(['en'])

# === 4. Output file ===
output_file = "detected_plates.txt"
open(output_file, "w").close()  # clear file at start

# === 5. Start webcam capture ===
cap = cv2.VideoCapture(0)  # change to 1/2 if multiple webcams

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO detection
    results = model(frame)

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy().astype(int)
        for (x_min, y_min, x_max, y_max) in boxes:
            # Draw bounding box
            cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

            # Crop plate
            plate_crop = frame[y_min:y_max, x_min:x_max]

            # OCR on plate
            ocr_result = reader.readtext(plate_crop)
            if ocr_result:
                plate_text = ocr_result[0][1].strip()
                cv2.putText(frame, plate_text, (x_min, y_min - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
                print("Detected Plate Text:", plate_text)

                # Save to file
                with open(output_file, "a") as f:
                    f.write(plate_text + "\n")

    cv2.imshow("License Plate Detection", frame)

    # Exit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

print(f"All detected plates saved to {output_file}")


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.



0: 480x640 (no detections), 168.6ms
Speed: 51.8ms preprocess, 168.6ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 144.5ms
Speed: 3.1ms preprocess, 144.5ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 132.0ms
Speed: 2.1ms preprocess, 132.0ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 137.7ms
Speed: 3.1ms preprocess, 137.7ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 128.1ms
Speed: 2.5ms preprocess, 128.1ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 171.6ms
Speed: 2.0ms preprocess, 171.6ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 126.3ms
Speed: 2.3ms preprocess, 126.3ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 115.0ms
Speed: 2.4ms prep

In [1]:
import cv2
print(cv2.__version__)
print(cv2.__file__)


4.12.0
C:\Users\H.T\AppData\Local\Programs\Python\Python311\Lib\site-packages\cv2\__init__.py
